In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [ ]:
df = pd.read_csv("academic_success.csv")
df.drop(columns=["id"], inplace=True)

In [ ]:
df.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,1,1,9238,1,1,126.0,1,1,19,...,0,6,7,6,12.428571,0,11.1,0.6,2.02,Graduate
1,1,17,1,9238,1,1,125.0,1,19,19,...,0,6,9,0,0.000000,0,11.1,0.6,2.02,Dropout
2,1,17,2,9254,1,1,137.0,1,3,19,...,0,6,0,0,0.000000,0,16.2,0.3,-0.92,Dropout
3,1,1,3,9500,1,1,131.0,1,19,3,...,0,8,11,7,12.820000,0,11.1,0.6,2.02,Enrolled
4,1,1,2,9500,1,1,132.0,1,19,37,...,0,7,12,6,12.933333,0,7.6,2.6,0.32,Graduate


In [ ]:
# Encode target
le = LabelEncoder()
df['Target'] = le.fit_transform(df['Target'])

# Split features and target
X = df.drop("Target", axis=1)
y = df["Target"]

# Fill missing values if any
X = X.fillna(X.mean(numeric_only=True))

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)
y_train_tensor = torch.LongTensor(y_train.values)
y_test_tensor = torch.LongTensor(y_test.values)

# DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)


## Define the Neural Network

In [ ]:
class AcademicSuccessModel(nn.Module):

    def __init__(self, input_dim, output_dim):
        super().__init__()

        # Input Layer -> Hidden Layer 1
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu1 = nn.ReLU()

        # Hidden Layer 1 -> Hidden Layer 2
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()

        # Hidden Layer 2 -> Output Layer
        self.fc3 = nn.Linear(64, output_dim)

        # Dropout
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):

        # First Hidden Layer
        sum_L1 = self.fc1(x)
        output_L1 = self.relu1(sum_L1)
        output_L1 = self.dropout(output_L1)

        # Second Hidden Layer
        sum_L2 = self.fc2(output_L1)
        output_L2 = self.relu2(sum_L2)
        output_L2 = self.dropout(output_L2)

        # Output Layer
        final_output = self.fc3(output_L2)

        return final_output


# Model parameters
input_dim = X_train.shape[1]
output_dim = len(np.unique(y))

# Instantiate model
model = AcademicSuccessModel(input_dim, output_dim)

print(model)

AcademicSuccessModel(
  (fc1): Linear(in_features=36, out_features=128, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=64, out_features=3, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


In [ ]:
input_dim, output_dim

(36, 3)

## Train the Model

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10
for epoch in range(epochs):
    for xb, yb in train_loader:
        outputs = model(xb)
        loss = loss_fn(outputs, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")


Epoch 10/10, Loss: 0.5907


## Evaluate the Model

In [ ]:
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. Evaluation Mode
model.eval()

# 2. Disable Gradient Calculation
with torch.no_grad():

    # 3. Predictions
    outputs = model(X_test_tensor)

    # 4. Get Predicted Classes
    y_pred = torch.argmax(outputs, dim=1)
    #    y_pred = outputs.round()

    # 5. Convert to NumPy
    y_pred = y_pred.cpu().numpy()
    y_true = y_test_tensor.cpu().numpy()

# 6. Evaluation Metrics
print("Accuracy:", accuracy_score(y_true, y_pred))


Accuracy: 0.8227914270778881
